In [1]:
import sys

sys.path.append("../")
from agents.trainers.ppo_trainer import PPOTrainer
import gymnasium as gym
import torch
from agent_configs.ppo_config import PPOConfig
from agent_configs.actor_config import ActorConfig
from agent_configs.critic_config import CriticConfig
from game_configs.cartpole_config import CartPoleConfig
from stats.stats import StatTracker

env = gym.make("CartPole-v1", render_mode="rgb_array")
config_dict = {
    "activation": "tanh",
    "clip_param": 0.2,
    "kernel_initializer": "orthogonal",
    # NORMALIZATION?
    "discount_factor": 0.99,
    "gae_lambda": 0.95,
    "critic_dense_layers": [64],
    "actor_dense_layers": [64],
    "actor_backbone": {
        "type": "dense",
        "widths": [64],
    },
    "critic_backbone": {
        "type": "dense",
        "widths": [64],
    },
    # REWARD CLIPPING
    "steps_per_epoch": 512,
    "train_policy_iterations": 4,
    "train_value_iterations": 4,
    "target_kl": 0.02,
    "entropy_coefficient": 0.01,
    "num_minibatches": 4,
    "loss_function": None,
    "training_steps": 500,
}

actor_config_dict = {
    "optimizer": torch.optim.Adam,
    "learning_rate": 2.5e-4,
    "adam_epsilon": 1e-7,
    "clipnorm": 0.5,
    "loss_function": None,
}

critic_config_dict = {
    "optimizer": torch.optim.Adam,
    "learning_rate": 2.5e-4,
    "adam_epsilon": 1e-7,
    "clipnorm": 0.5,
    "loss_function": None,
}

print("Actor Config")
actor_config = ActorConfig(actor_config_dict)
print("Critic Config")
critic_config = CriticConfig(critic_config_dict)

print("PPO Config")
config = PPOConfig(
    config_dict,
    CartPoleConfig(),
    actor_config=actor_config,
    critic_config=critic_config,
)

trainer = PPOTrainer(
    config,
    env,
    torch.device("cpu"),
    StatTracker("ppo"),
)

trainer.checkpoint_interval = 100
trainer.test_interval = 100
trainer.test_trials = 50

trainer.train()

Actor Config
Using         adam_epsilon                  : 1e-07
Using         learning_rate                 : 0.00025
Using         clipnorm                      : 0.5
Using         optimizer                     : <class 'torch.optim.adam.Adam'>
Critic Config
Using         adam_epsilon                  : 1e-07
Using         learning_rate                 : 0.00025
Using         clipnorm                      : 0.5
Using         optimizer                     : <class 'torch.optim.adam.Adam'>
PPO Config
Using default save_intermediate_weights     : False
Using         training_steps                : 500
Using default adam_epsilon                  : 1e-08
Using default momentum                      : 0.9
Using default learning_rate                 : 0.001
Using default clipnorm                      : 0
Using default optimizer                     : <class 'torch.optim.adam.Adam'>
Using default weight_decay                  : 0.0
Using         num_minibatches               : 4
Using default 

In [2]:
import gymnasium as gym
import sys

sys.path.append("../")
from agents.trainers.rainbow_trainer import RainbowTrainer
from game_configs.cartpole_config import CartPoleConfig
from agent_configs.dqn.rainbow_config import RainbowConfig
from losses.basic_losses import KLDivergenceLoss
from stats.stats import StatTracker
import torch

# env = ClipReward(AtariPreprocessing(gym.make("MsPacmanNoFrameskip-v4", render_mode="rgb_array"), terminal_on_life_loss=True), -1, 1) # as recommended by the original paper, should already include max pooling
# env = FrameStack(env, 4)
env = gym.make("CartPole-v1")


config_dict = {
    "backbone": {
        "type": "dense",
        "widths": [128],
    },
    "adam_epsilon": 1e-8,
    "learning_rate": 0.001,
    "training_steps": 5000,
    "per_epsilon": 1e-6,
    "per_alpha": 0.2,
    "per_beta": 0.6,
    "minibatch_size": 128,
    "transfer_interval": 100,
    "n_step": 3,
    "noisy_sigma": 0.5,
    "replay_interval": 1,
    "kernel_initializer": "orthogonal",
    "noisy_sigma": 0.5,
    "loss_function": KLDivergenceLoss(),  # could do categorical cross entropy
    "atom_size": 51,
    "clipnorm": 10.0,
    "model_name": "rainbow_refactor",
}
game_config = CartPoleConfig()
config = RainbowConfig(config_dict, game_config)
trainer = RainbowTrainer(
    config, env, torch.device("cpu"), StatTracker("rainbow_refactor")
)

trainer.checkpoint_interval = 100
trainer.test_interval = 500

trainer.train()

Using default save_intermediate_weights     : False
Using         training_steps                : 5000
Using         adam_epsilon                  : 1e-08
Using default momentum                      : 0.9
Using         learning_rate                 : 0.001
Using         clipnorm                      : 10.0
Using default optimizer                     : <class 'torch.optim.adam.Adam'>
Using default weight_decay                  : 0.0
Using default num_minibatches               : 1
Using default training_iterations           : 1
Using default lr_schedule_type              : none
Using default lr_schedule_steps             : []
Using default lr_schedule_values            : []
Using         minibatch_size                : 128
Using default replay_buffer_size            : 5000
Using default min_replay_buffer_size        : 128
Using         n_step                        : 3
Using default discount_factor               : 0.99
Using         per_alpha                     : 0.2
Using         per_b

In [ ]:
import sys

sys.path.append("../")
import time
import torch
from losses.basic_losses import CategoricalCrossentropyLoss, KLDivergenceLoss
from agents.random import RandomAgent

# from agents.muzero import MuZeroAgent
from agents.trainers.muzero_trainer import MuZeroTrainer
from agent_configs.muzero_config import MuZeroConfig
from game_configs.tictactoe_config import TicTacToeConfig
from agents.tictactoe_expert import TicTacToeBestAgent
from modules.world_models.muzero_world_model import MuzeroWorldModel
from stats.stats import StatTracker

# Ensure we use CPU for fairness/comparibility or GPU if available
device = "cpu"  # or "cuda" if available
print(f"Using device: {device}")

params = {
    "num_simulations": 25,
    "per_alpha": 0.0,
    "per_beta": 0.0,
    "per_beta_final": 0.0,
    "n_step": 10,
    "root_dirichlet_alpha": 0.25,
    "residual_layers": [(24, 3, 1)],
    "reward_dense_layer_widths": [],
    "reward_conv_layers": [(16, 1, 1)],
    "actor_dense_layer_widths": [],
    "actor_conv_layers": [(16, 1, 1)],
    "critic_dense_layer_widths": [],
    "critic_conv_layers": [(16, 1, 1)],
    "to_play_dense_layer_widths": [],
    "to_play_conv_layers": [(16, 1, 1)],
    "backbone": {
        "type": "resnet",
        "filters": [24],
        "kernel_sizes": [3],
        "strides": [1],
    },
    "reward_backbone": {
        "type": "resnet",
        "filters": [16],
        "kernel_sizes": [1],
        "strides": [1],
    },
    "to_play_backbone": {
        "type": "resnet",
        "filters": [16],
        "kernel_sizes": [1],
        "strides": [1],
    },
    "actor_backbone": {
        "type": "resnet",
        "filters": [16],
        "kernel_sizes": [1],
        "strides": [1],
    },
    "critic_backbone": {
        "type": "resnet",
        "filters": [16],
        "kernel_sizes": [1],
        "strides": [1],
    },
    "known_bounds": [-1, 1],
    "support_range": None,
    "minibatch_size": 8,
    "replay_buffer_size": 100000,
    "gumbel": False,
    "gumbel_m": 16,
    "policy_loss_function": CategoricalCrossentropyLoss(),
    "training_steps": 10000,  # Reduced for benchmark speed
    "transfer_interval": 1,
    "num_workers": 4,
    "world_model_cls": MuzeroWorldModel,
    "search_batch_size": 0,  # Iterative
    "use_virtual_mean": False,
    "virtual_loss": 3.0,
}

game_config = TicTacToeConfig()

print("--- Running MuZero Comaprison of Changes ---")
params_batched = params.copy()
params_batched["search_batch_size"] = 5
params_batched["use_virtual_mean"] = True
params_batched["model_name"] = "muzero_search_refactor"

env_batch = TicTacToeConfig().make_env()
config_batch = MuZeroConfig(config_dict=params_batched, game_config=game_config)
config_batch.search_batch_size = 5  # Explicitly set

trainer = MuZeroTrainer(
    config_batch,
    env_batch,
    torch.device("cpu"),
    stats=StatTracker(model_name="muzero_search_refactor"),
    test_agents=[RandomAgent(), TicTacToeBestAgent()],
)
trainer.checkpoint_interval = 100
trainer.test_interval = 1000
trainer.test_trials = 100

start_time = time.time()
trainer.train()
end_time = time.time()
print(f"MuZero Batched Time: {end_time - start_time:.2f}s")

Using device: cpu
--- Running MuZero Comaprison of Changes ---
Using default save_intermediate_weights     : False
Using         training_steps                : 20000
Using default adam_epsilon                  : 1e-08
Using default momentum                      : 0.9
Using default learning_rate                 : 0.001
Using default clipnorm                      : 0
Using default optimizer                     : <class 'torch.optim.adam.Adam'>
Using default weight_decay                  : 0.0
Using default num_minibatches               : 1
Using default training_iterations           : 1
Using default lr_schedule_type              : none
Using default lr_schedule_steps             : []
Using default lr_schedule_values            : []
Using         minibatch_size                : 8
Using         replay_buffer_size            : 100000
Using default min_replay_buffer_size        : 8
Using         n_step                        : 10
Using default discount_factor               : 0.99
Using    

KeyboardInterrupt: 

Process Process-4:
Process Process-3:
Process Process-2:
Process Process-1:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/test_notebooks/../agents/executors/torch_mp_executor.py", line 45, in _worker_loop
    data = worker.play_sequence()
           ^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/test_notebooks/../agents/actors/actors.py", line 175, in play_sequence
    transition = self.step()
                 ^^^^^^^^^^^
  File "/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/test_notebooks/../agents/actors/actors.py", line 123, in step
    action = self.policy.compute_action(


In [1]:
import sys

sys.path.append("../")
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
import random

from custom_gym_envs.envs.matching_pennies import (
    env as matching_pennies_env,
    MatchingPenniesGymEnv,
)
from agents.trainers.nfsp_trainer import NFSPTrainer
from agent_configs.dqn.nfsp_config import NFSPDQNConfig
from agent_configs.dqn.rainbow_config import RainbowConfig
from game_configs.matching_pennies_config import MatchingPenniesConfig
from losses.basic_losses import (
    KLDivergenceLoss,
    CategoricalCrossentropyLoss,
    HuberLoss,
    MSELoss,
)
import torch
from torch.optim import Adam, SGD
from stats.stats import StatTracker

config_dict = {
    "shared_networks_and_buffers": False,
    "training_steps": 10000,
    "anticipatory_param": 0.1,
    "replay_interval": 2,
    "num_minibatches": 1,
    "learning_rate": 0.1,
    "momentum": 0.0,
    "optimizer": SGD,
    "loss_function": HuberLoss(),
    "min_replay_buffer_size": 500,
    "minibatch_size": 128,
    "replay_buffer_size": 1000,
    "transfer_interval": 300,
    # --- NEW RL BACKBONE ---
    "backbone": {"type": "dense", "widths": [128]},
    # Note: value_backbone and advantage_backbone default to Identity (None)
    # which replaces the old empty list [] behavior.
    "noisy_sigma": 0.0,
    "eg_epsilon": 0.06,
    "eg_epsilon_decay_type": "inverse_sqrt",
    "eg_epsilon_decay_final_step": 0,
    # --- SL CONFIG ---
    "sl_learning_rate": 0.01,
    "sl_momentum": 0.0,
    "sl_optimizer": SGD,
    "sl_loss_function": CategoricalCrossentropyLoss(),
    "sl_min_replay_buffer_size": 500,
    "sl_minibatch_size": 128,
    "sl_replay_buffer_size": 20000,
    # --- NEW SL BACKBONE ---
    "sl_backbone": {"type": "dense", "widths": [128]},
    "sl_clip_low_prob": 0.0,
    "per_alpha": 0.0,
    "per_beta": 0.0,
    "per_beta_final": 0.0,
    "per_epsilon": 0.00001,
    "n_step": 1,
    "atom_size": 1,
    "dueling": False,
    "clipnorm": 10.0,
    "sl_clipnorm": 10.0,
}

config = NFSPDQNConfig(
    config_dict=config_dict,
    game_config=MatchingPenniesConfig(),
)
config.save_intermediate_weights = True

import custom_gym_envs
import gymnasium as gym

# env = gym.make("custom_gym_envs/LeducHoldem-v0", encode_player_turn=False)

env = matching_pennies_env(render_mode="rgb_array", max_cycles=1)

trainer = NFSPTrainer(config, env, torch.device("cpu"), StatTracker("nfsp_3"))
trainer.checkpoint_interval = 100
trainer.test_interval = 1000
trainer.test_trials = 10000
trainer.train()

NFSPDQNConfig
Using default save_intermediate_weights     : False
Using         training_steps                : 10000
Using default adam_epsilon                  : 1e-08
Using         momentum                      : 0.0
Using         learning_rate                 : 0.1
Using         clipnorm                      : 10.0
Using         optimizer                     : <class 'torch.optim.sgd.SGD'>
Using default weight_decay                  : 0.0
Using         num_minibatches               : 1
Using default training_iterations           : 1
Using default lr_schedule_type              : none
Using default lr_schedule_steps             : []
Using default lr_schedule_values            : []
Using         minibatch_size                : 128
Using         replay_buffer_size            : 1000
Using         min_replay_buffer_size        : 500
Using         n_step                        : 1
Using default discount_factor               : 0.99
Using         per_alpha                     : 0.0
Using   

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment custom_gym_envs/Catan-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment custom_gym_envs/MatchingPennies-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment custom_gym_envs/Connect4-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/gymnasium/envs/

TypeError: must be real number, not NoneType